In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, BatchNormalization, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
from tensorflow.keras.callbacks import ModelCheckpoint


In [ ]:
%run Data_Setup.ipynb


In [ ]:
backbone = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(CONFIGURATION["IM_SIZE"], CONFIGURATION["IM_SIZE"], 3)
)
backbone.trainable = False

inputs = Input(shape=(CONFIGURATION["IM_SIZE"], CONFIGURATION["IM_SIZE"], 3))
x = backbone(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)
outputs = Dense(CONFIGURATION["NUM_CLASSES"], activation="softmax")(x)

model = Model(inputs, outputs)
model.summary()


In [ ]:
model.compile(
    optimizer=Adam(CONFIGURATION["LEARNING_RATE"]),
    loss=CategoricalCrossentropy(),
    metrics=[CategoricalAccuracy(name="accuracy")]
)

checkpoint = ModelCheckpoint("../models/mobilenet_best.keras", monitor="val_accuracy", save_best_only=True)

history = model.fit(training_dataset, validation_data=validation_dataset, epochs=CONFIGURATION["N_EPOCHS"], callbacks=[checkpoint])


In [ ]:
model.load_weights("../models/mobilenet_best.keras")
model.evaluate(validation_dataset)


In [ ]:
lenet_model.save("lenet_full_model.keras")

In [ ]:
from google.colab import files
files.download("lenet_full_model.keras")